# Analysis for the counter-zero tests

**Deliver notebook, this runs with Restart Kernel and Run All Cells**

### This is to deal with MWorks skipping counter zero after code changes in ngfriedman/2310/imaginggate-laseroff 

- files in data/241111-countertest-issue[123].mwk2
- testing mwkfiles.py reader code

### references
- See mworksbehavior issue #52
- see MH LabArchives notes

# Logic of changes
- see details in MH LabArchives note, '250119: changing mwkfiles.py to deal with counter issues'





## Set up code

In [1]:
%autoreload 2

from pathlib import Path
import sys, os
from types import SimpleNamespace
import pandas as pd 

import mworksbehavior as mwb
from mworksbehavior import mwkfiles

## check to make sure the mwkfiles code is coming from the right place
# there may be better ways to do this, by forcing mwb directory to
# start of sys.path, for ex
packdir = Path('~/Repositories/mworksbehavior/mworksbehavior').expanduser()
# if this assertion fails you may need cd ~/Repositories/mworksbehavior; pip -e .
assert packdir.resolve() == Path(mwb.__path__[0]).resolve(), \
  'does not look like mworksbehavior is installed in editable form at ~/Repositories/mworksbehavior'

## Select one of the three mwk2 files here. Paul assembled a set of tests for these too in test_mwkfiles.py

In [2]:
datdir = Path('~/Repositories/mworksbehavior/mworksbehavior/tests/data').expanduser()

#mwkname = f'240918-countertest-issue1.mwk2'
mwkname = f'240918-countertest-issue2.mwk2'
#mwkname = f'241111-countertest-issue3.mwk2'


### Read the file using the plain MWF code and display the codestream for debugging

In [3]:
# Read the file using the plain MWF code
fulln = datdir / mwkname

mwf = mwkfiles.MWKFile(fulln);


In [4]:
print(mwf.firstTrStartTimeUs)
print(mwf.lastTrEndTimeUs)

262011259
330127328


In [5]:
counterIx = mwf.df.tagname.isin(['stimPySelLevel'])
#print(mwf.df.loc[counterIx,:])

In [6]:
## quick counterFIO analysis from raw stream
counterIx = mwf.df.tagname.isin(['counterFIO'])
cdf = mwf.df.loc[counterIx,:]
np.unique(np.diff(cdf.value))

array([-2244, 0, 1], dtype=object)

In [7]:
counterIx = mwf.df.tagname.isin(['counterFIO','strobedDigitalWord', 'stimPySelLevel'])
pd.set_option('display.max_rows', 60)
pd.set_option('display.min_rows', 50)


mwf.df.loc[counterIx,:]

,tagname,timeUs,value,timeFmStUs
159,stimPySelLevel,260282108,2,129
197,strobedDigitalWord,260282124,4,145
199,counterFIO,260282124,2245,145
363,stimPySelLevel,260284031,2,2052
401,counterFIO,260284135,2245,2156
402,strobedDigitalWord,260284136,4,2157
574,stimPySelLevel,262000699,2,1718720
612,counterFIO,262000750,2245,1718771
613,strobedDigitalWord,262000750,4,1718771
637,strobedDigitalWord,262009570,0,1727591


## Test with the full mwkfiles.RetinotopyMap2Stim / CounterStimMixin parsing code

In [8]:
mwf = mwkfiles.RetinotopyMap2StimMWKFile(fulln,doTryFixCorrupt=True)
mwf.compute_imaging_constants()
print(mwf.nstim, mwf.nframes_stim)
assert mwf.nstim * mwf.nframes_stim == 1920

8 240


/Users/histed/Repositories/mworksbehavior/mworksbehavior/mwkfiles.py:378: UserWarning: Found first counter tick to be 1, expected zero; subtracting 1 from all counter vals, altering df values
  warnings.warn(f"Found first counter tick to be 1, expected zero; subtracting 1 from all counter vals, altering df values")
/Users/histed/Repositories/mworksbehavior/mworksbehavior/mwkfiles.py:470: UserWarning: Trying to fix file: Number of frames is one less than expected, probably first trial is short, be careful w/ analysis
  warnings.warn('Trying to fix file: Number of frames is one less than expected, probably first trial is short, be careful w/ analysis')


In [9]:
display(mwf.counterStats)

namespace(nCounterTicks=1919, nTotalStims=8, ticksPerStim=239.875)

In [10]:
display(mwf.firstCounterUs)
display(mwf.firstTrStartTimeUs)
display(mwf.firstTrEndTimeUs)
display(mwf.constS)

tdf = mwf.get_codedf(mwf.counterVar)
#display(tdf)

display(mwf.stimDf)

counterStatN = SimpleNamespace(minCounterVal=min(tdf.value),
                               maxCounterVal=max(tdf.value), 
                               nTotalStim=len(mwf.stimDf),
                               nUniqueStim=mwf.nstim)
ctSN = counterStatN
ctSN.counterRange = ctSN.maxCounterVal - ctSN.minCounterVal + 2 # count at both ends
ctSN.counterTickPerStim = ctSN.counterRange/ctSN.nTotalStim
display(ctSN)

0    266196329
Name: timeUs, dtype: int64

262011259

274126555

counterNPre      60
counterNPost    120
counterNStim     60
dtype: int64

,stimPySelLevel,tStim1AzimuthDeg,tStim2AzimuthDeg,tStim1ElevationDeg,tStim2ElevationDeg,tStim1GratingSpeedDps,tStim2GratingSpeedDps,tStim1GratingSpatialFreqCpd,tStim2GratingSpatialFreqCpd,tStim1OriNoiseF0Cpd,...,tATrainNPulses,tBTrainNPulses,tATrainPulseLengthMs,tBTrainPulseLengthMs,tARampLengthMs,tBRampLengthMs,tARampExtraConstantLengthMs,tBRampExtraConstantLengthMs,tAStartOffsetMs,tBStartOffsetMs
0,2,0,0,0,0,20,0,0.1,0.1,0.1,...,2,2,10,100,2,2,0,0,0,0
1,5,0,0,0,0,20,0,0.1,0.1,0.1,...,2,2,10,100,2,2,0,0,0,0
2,6,0,0,0,0,20,0,0.1,0.1,0.1,...,2,2,10,100,2,2,0,0,0,0
3,7,0,0,0,0,20,0,0.1,0.1,0.1,...,2,2,10,100,2,2,0,0,0,0
4,0,0,0,0,0,20,0,0.1,0.1,0.1,...,2,2,10,100,2,2,0,0,0,0
5,4,0,0,0,0,20,0,0.1,0.1,0.1,...,2,2,10,100,2,2,0,0,0,0
6,3,0,0,0,0,20,0,0.1,0.1,0.1,...,2,2,10,100,2,2,0,0,0,0
7,1,0,0,0,0,20,0,0.1,0.1,0.1,...,2,2,10,100,2,2,0,0,0,0


namespace(minCounterVal=1,
          maxCounterVal=1919,
          nTotalStim=8,
          nUniqueStim=8,
          counterRange=1920,
          counterTickPerStim=240.0)